<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

# 학습 내용
>이번 장에서는 <strong>Planner-Worker 패턴(Planner-Worker Pattern)</strong>에 대해 학습합니다.
>계획을 세우는 Planner와 실행하는 Worker를 분리한 멀티 에이전트 패턴을 학습해봅시다.

# Planner-Worker 패턴 (Planner-Worker Pattern)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">계획(Plan)과 실행(Execute)을 분리</mark>하여 복잡한 작업을 체계적으로 수행하는 멀티 에이전트 패턴입니다.

"고객 불만 분석 보고서를 작성해줘"처럼 복잡한 작업을 단일 LLM 호출로 처리하면 한계가 있습니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">다단계 작업</mark>은 데이터 수집 → 분류 → 분석 → 작성 순서로 진행되어야 하는데, 하나의 프롬프트로는 각 단계를 충분히 깊게 수행하기 어렵습니다. 계획과 실행을 분리하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">품질 향상</mark>(Planner가 전체 구조를 먼저 잡고 Worker는 각 단계에 집중), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">비용 최적화</mark>(<mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">상위 모델(Planner)</mark>은 전략적 판단에만 사용하고 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">하위 모델(Worker)</mark>은 반복 실행에 사용하여 API 비용 절감), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">유연성</mark>(Reflection 단계에서 계획을 재수립하거나 Worker를 교체하는 것이 용이)의 이점을 얻을 수 있습니다.</br>
이 내용을 학습하기 전에 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Agent 구성요소</mark>(LLM, Tool, Planning, Memory의 역할, Ch.4-2-1_001 참고), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">StateGraph</mark>(노드와 엣지로 워크플로우를 구조화하는 방법, Ch.4-2-1_002 참고), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LLM API</mark>(모델별 성능/비용 차이)의 개념을 먼저 이해하면 좋습니다.

## AgentState 정의
> 멀티 에이전트가 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">공유하는 상태</mark>를 정의합니다.

In [ ]:
# TODO 1: 상태 타입을 정의하세요. 필드는 task(str), plan(str), results(리스트 누적 방식), reflection(str), final_answer(str)입니다. 필드 목록을 출력하세요.

from typing import TypedDict, List, Annotated
import operator

class AgentState(TypedDict):
    task: str                              # 원래 작업
    plan: str                              # Planner가 생성한 계획
    results: Annotated[List[str], operator.add]  # Worker 결과 누적
    reflection: str                        # 반성/평가
    final_answer: str                      # 최종 답변

print(f"AgentState 필드: {list(AgentState.__annotations__.keys())}")

💡Annotated + operator.add
> `Annotated[List[str], operator.add]`는 상태 업데이트 시 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">기존 리스트에 추가</mark>됩니다.
> `return {"results": ["새 결과"]}`하면 기존 results에 append됩니다.

## 노드 구현

In [ ]:
# TODO 2: planner, worker, reflection 3개의 노드 함수를 정의하세요. planner는 작업을 분석하여 계획을 반환하고, worker는 계획에 따라 실행 결과를 반환하고, reflection은 결과를 평가하여 반환합니다. 테스트 state로 planner를 실행하여 확인하세요.

def planner(state: AgentState) -> AgentState:
    """작업을 분석하고 실행 계획 수립"""
    prompt = f"다음 작업의 실행 계획을 세워주세요: {state['task']}"
    plan = llm.invoke(prompt).content
    return {"plan": plan}

def worker(state: AgentState) -> AgentState:
    """계획에 따라 작업 실행"""
    prompt = f"계획: {state['plan']}\n실행하세요."
    result = llm.invoke(prompt).content
    return {"results": [result]}

def reflection(state: AgentState) -> AgentState:
    """결과를 평가하고 개선점 도출"""
    prompt = f"결과: {state['results']}\n품질을 평가하세요."
    review = llm.invoke(prompt).content
    return {"reflection": review}

# 테스트: 각 노드 단독 실행
test_state = {"task": "고객 불만 분석 보고서 작성", "results": []}
plan_result = planner(test_state)
print(f"계획 수립 완료: {plan_result['plan'][:80]}...")

## 그래프 조립

In [ ]:
# TODO 3: 평가 분기 함수를 정의하고 (reflection에 "개선"이 있으면 "planner", 없으면 종료 반환), 상태 그래프를 조립하세요. 흐름은 START→planner→worker→reflection→(조건부: planner 또는 종료)입니다. "고객 불만 분석 보고서 작성"으로 실행하여 결과를 출력하세요.

def evaluate(state: AgentState) -> str:
    """개선 필요 여부 판단"""
    if "개선" in state["reflection"]:
        return "planner"
    return "end"

graph = StateGraph(AgentState)
graph.add_node("planner", planner)
graph.add_node("worker", worker)
graph.add_node("reflection", reflection)

graph.add_edge(START, "planner")
graph.add_edge("planner", "worker")
graph.add_edge("worker", "reflection")
graph.add_conditional_edges("reflection", evaluate,
    {"planner": "planner", "end": END})

app = graph.compile()
result = app.invoke({"task": "고객 불만 분석 보고서 작성", "results": []})
print(f"실행 결과 수: {len(result['results'])}")
print(f"최종 반성: {result['reflection'][:80]}...")

💡Planner-Worker의 장점
> Planner가 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">전체 작업을 분해</mark>하므로 Worker는 단순 실행에 집중합니다.
> Reflection 단계에서 품질을 검증하고 반복 개선이 가능합니다.